# B3: Progress Indicators

---

## Overview

Track construction progress through inspection milestones.

**Inspection Sequence:**
1. Foundation
2. Framing/Rough
3. Electrical/Plumbing/Mechanical Rough
4. Insulation
5. Drywall
6. Final inspections

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

# Add modules to path
sys.path.insert(0, str(Path.cwd().parent))

from modules.timeline_calculator import (
    calculate_progress_percent,
    identify_stalled_projects,
    INSPECTION_SEQUENCE
)
from modules.data_loader import load_csv

# Configuration
with open('../00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

DATA_DIR = Path(CONFIG['paths']['data_dir'])
STALLED_THRESHOLD = CONFIG['timeline']['stalled_days_threshold']

print("Inspection Sequence:")
for i, insp in enumerate(INSPECTION_SEQUENCE, 1):
    pct = round(100 * i / len(INSPECTION_SEQUENCE), 1)
    print(f"  {i:2}. {insp:20} ({pct}% complete)")

## 2. Load Project Data

In [ ]:
# Load classified projects
housing_path = DATA_DIR / 'housing_projects_classified.csv'
if not housing_path.exists():
    housing_path = Path(CONFIG['paths']['housing_projects'])

df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    display(df[['address_display', 'status', 'net_units']].head(10))

## 3. Calculate Progress Percentage

In [ ]:
# Example progress calculation
test_inspections = [
    ['Foundation'],
    ['Foundation', 'Framing/Rough'],
    ['Foundation', 'Framing', 'Electrical Rough', 'Plumbing Rough'],
    ['Foundation', 'Framing', 'Drywall'],
    INSPECTION_SEQUENCE,  # All complete
]

print("Progress Calculation Examples:")
print("="*60)

for inspections in test_inspections:
    pct = calculate_progress_percent(inspections)
    print(f"  {inspections[:3]}{'...' if len(inspections) > 3 else ''}: {pct}% complete")

## 4. Identify Stalled Projects

Projects with no activity for 180+ days.

In [ ]:
# Check for stalled projects
print(f"Stalled threshold: {STALLED_THRESHOLD} days")

# TODO: This requires last_action_date column from permit data
# When that data is available:
# stalled = identify_stalled_projects(df, STALLED_THRESHOLD, 'last_action_date')

# For now, identify projects in 'In Review' status (may be stalled)
if df is not None and 'status' in df.columns:
    review_status = ['In Review', 'Under Review', 'Incomplete Pending Applicant']
    potentially_stalled = df[df['status'].isin(review_status)]
    
    print(f"\nProjects potentially stalled (in review status): {len(potentially_stalled)}")
    print(f"Total units in review: {potentially_stalled['net_units'].sum():,.0f}")
    
    if len(potentially_stalled) > 0:
        print("\nLargest projects in review:")
        display(potentially_stalled.nlargest(10, 'net_units')[['address_display', 'net_units', 'status']])

## 5. Projects Near Completion

In [ ]:
# Identify projects with approved/permitted status (likely under construction)
if df is not None and 'status' in df.columns:
    approved_keywords = ['Approved', 'Pending Final Action', 'Final']
    
    near_completion = df[df['status'].str.contains('|'.join(approved_keywords), case=False, na=False)]
    
    print(f"Projects approved/near completion: {len(near_completion)}")
    print(f"Total units: {near_completion['net_units'].sum():,.0f}")
    
    if len(near_completion) > 0:
        print("\nLargest approved projects:")
        display(near_completion.nlargest(10, 'net_units')[['address_display', 'net_units', 'status']])

## 6. Export Progress Data

In [ ]:
# Export progress summary
if df is not None:
    # Add progress category
    def get_progress_category(status):
        status = str(status).lower()
        if 'complete' in status or 'occupancy' in status:
            return 'Completed'
        elif 'approved' in status or 'final action' in status:
            return 'Near Completion'
        elif 'review' in status:
            return 'In Progress'
        else:
            return 'Early Stage'
    
    df['progress_category'] = df['status'].apply(get_progress_category)
    
    output_path = DATA_DIR / 'housing_projects_progress.csv'
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

---

## Summary

This notebook:
- Defined inspection-based progress calculation
- Identified potentially stalled projects
- Tracked projects near completion

**Next:** Run `C1_pipeline_analysis.ipynb` for comprehensive analysis.